# 1. Install Dependencies

In [3]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 7.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01


# 2. Setup Environment

## Import Libraries

In [4]:
import os
import json
import glob
import torch

import pandas as pd
import numpy as np
from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [5]:
print("PyTorch version :", torch.__version__)
print("GPUs available  :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB") 

PyTorch version : 2.9.0+cu126
GPUs available  : 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


# 3. Config

In [6]:
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
HF_REPO_ID = "somendrew/genz-qwen2.5-1.5b"
 
DATA_DIR   = "/kaggle/input/datasets/somendrew/genz-response-dataset"
DATA_FILES = sorted(glob.glob(os.path.join(DATA_DIR, "batch*.json")))
 
OUTPUT_DIR  = "/kaggle/working/genz-adapter"
MAX_SEQ_LEN = 512
EPOCHS      = 3
BATCH_SIZE  = 4    # per device — 4 × 2 GPUs = 8 per step
GRAD_ACCUM  = 2    # effective global batch = 16
LR          = 2e-4
 
print(f"📂 Found {len(DATA_FILES)} data files:")
for f in DATA_FILES:
    print(f"   {f}")

📂 Found 10 data files:
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch10_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch1_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch2_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch3_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch4_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch5_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch6_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch7_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch8_fixed.json
   /kaggle/input/datasets/somendrew/genz-response-dataset/batch9_fixed.json


# 4. Prepare Dateset

## Load 

In [7]:
files   = sorted(glob.glob("/kaggle/input/datasets/somendrew/genz-response-dataset/batch*.json"))
records = []

for f in files:
    records += json.load(open(f))
    print(f"✅ {f.split('/')[-1]} → {len(records)} total so far")

print(f"\n📦 Total: {len(records)} records")

✅ batch10_fixed.json → 100 total so far
✅ batch1_fixed.json → 200 total so far
✅ batch2_fixed.json → 296 total so far
✅ batch3_fixed.json → 396 total so far
✅ batch4_fixed.json → 496 total so far
✅ batch5_fixed.json → 596 total so far
✅ batch6_fixed.json → 696 total so far
✅ batch7_fixed.json → 791 total so far
✅ batch8_fixed.json → 877 total so far
✅ batch9_fixed.json → 908 total so far

📦 Total: 908 records


## Format & Split

In [8]:
#format
def format_example(r):
    user = r["instruction"]
    if r.get("input"):
        user += "\n\n" + r["input"]
    return {
        "text": (
            f"<|im_start|>system\n"
            f"You are a GenZ assistant. Reply using GenZ slang and emojis. No formal language, just vibes fr 🔥\n"
            f"<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{r['output']}<|im_end|>"
        )
    }

formatted = [format_example(r) for r in records]

#Split Dataset
dataset = Dataset.from_list(formatted)
split   = dataset.train_test_split(test_size=0.1, seed=42)

print(f"✅ Train: {len(split['train'])}  |  Val: {len(split['test'])}")

✅ Train: 817  |  Val: 91


In [9]:
dataset

Dataset({
    features: ['text'],
    num_rows: 908
})

In [10]:
split

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 817
    })
    test: Dataset({
        features: ['text'],
        num_rows: 91
    })
})

In [11]:
split['train']

Dataset({
    features: ['text'],
    num_rows: 817
})

In [12]:
split['train'][0]

{'text': '<|im_start|>system\nYou are a GenZ assistant. Reply using GenZ slang and emojis. No formal language, just vibes fr 🔥\n<|im_end|>\n<|im_start|>user\nExplain why the given definition is wrong.\n\nA mole is an animal that lives underground.<|im_end|>\n<|im_start|>assistant\nIncomplete – misses mammal Talpidae + confuses with chem mole 🧪. Sus def or science just facepalmed 😩😂<|im_end|>'}

# 5. Load Model & Tokenizer

In [13]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model dtype  : {model.dtype}")
print(f"Device map   : {model.hf_device_map}")
print(f"Pad token    : {tokenizer.pad_token}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model dtype  : torch.bfloat16
Device map   : {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 1, 'model.layers.2': 1, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1}
Pad token    : <|im_end|>


# 6. Configure Lora

In [14]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# 7. Train with SFTTrainer

In [15]:
import trl
print(trl.__version__)

0.29.0


In [16]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    peft_config=lora_config,
    formatting_func=lambda x: x["text"],
    args=SFTConfig(                          # ← SFTConfig instead of TrainingArguments
        output_dir="/kaggle/working/genz-adapter",
        max_length=512,                  # ← now lives here
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=2,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
    ),
)

Applying formatting function to train dataset:   0%|          | 0/817 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/817 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/817 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/817 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

In [17]:
print(trainer.state.global_step)   
print(trainer.state.epoch)         

0
None


In [18]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,1.390812,1.302319
200,1.050728,1.281843
300,0.877140,1.362948


TrainOutput(global_step=309, training_loss=1.1795992527193235, metrics={'train_runtime': 1598.0398, 'train_samples_per_second': 1.534, 'train_steps_per_second': 0.193, 'total_flos': 2229610513222656.0, 'train_loss': 1.1795992527193235})

In [19]:
print(trainer.state.global_step)   
print(trainer.state.epoch)         

309
3.0


# Save Adapters

In [20]:
trainer.model.save_pretrained("/kaggle/working/genz-adapter")
tokenizer.save_pretrained("/kaggle/working/genz-adapter")
print("✅ Adapter saved!")

✅ Adapter saved!


# Merge and Save

In [22]:
trainer.model.config.use_cache = True
merged_model = trainer.model.merge_and_unload()

# save directly — no .to(bfloat16)
merged_model.save_pretrained(
    "/kaggle/working/genz-merged",
    safe_serialization=True,
    max_shard_size="5GB",
)
tokenizer.save_pretrained("/kaggle/working/genz-merged")

# check file size
for f in os.listdir("/kaggle/working/genz-merged"):
    size = os.path.getsize(f"/kaggle/working/genz-merged/{f}") / 1024**2
    print(f"  {f}  ({size:.1f} MB)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  config.json  (0.0 MB)
  chat_template.jinja  (0.0 MB)
  generation_config.json  (0.0 MB)
  model.safetensors  (1090.4 MB)
  tokenizer.json  (10.9 MB)
  tokenizer_config.json  (0.0 MB)


#### model.safetensors is still ~1GB — still saving in 4-bit. The merge isn't dequantizing properly.
#### We need to load the base model fresh in bfloat16 and manually apply the adapter on top:

## Merge on base model

In [23]:
from peft import PeftModel

# Step 1 — load base model fresh in bfloat16 (no quantization)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Step 2 — load adapter on top of clean base model
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/genz-adapter",
)

# Step 3 — merge and save
merged = model_with_adapter.merge_and_unload()
merged.save_pretrained("/kaggle/working/genz-merged-v2", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/genz-merged-v2")

# check size — should be ~3000 MB now
for f in os.listdir("/kaggle/working/genz-merged-v2"):
    size = os.path.getsize(f"/kaggle/working/genz-merged-v2/{f}") / 1024**2
    print(f"  {f}  ({size:.1f} MB)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  config.json  (0.0 MB)
  chat_template.jinja  (0.0 MB)
  generation_config.json  (0.0 MB)
  model.safetensors  (2944.4 MB)
  tokenizer.json  (10.9 MB)
  tokenizer_config.json  (0.0 MB)


# 8. Evaluate

In [25]:
base_model.eval()
base_model.config.use_cache = True
SYSTEM_PROMPT = "You are a GenZ assistant. Reply using GenZ slang and emojis. No formal language, just vibes fr 🔥"
def chat(prompt):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(text, return_tensors="pt").to(merged.device)
    with torch.no_grad():
        output = merged.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(chat("What is gravity?"))
print(chat("Explain machine learning."))

Gravity = universal glue holding planets n stars together 🌌. Or stuff falls down – basic af
ML computers learn like humans from data 📊 n make predictions future stuff. AI said 'bet' 😂


# 9. Save Adapter & Model

In [34]:
# Save LoRA adapter
trainer.model.save_pretrained("/kaggle/working/genz-adapter")
tokenizer.save_pretrained("/kaggle/working/genz-adapter")
print("✅ Adapter saved")

# Merge adapter into base model and save full model
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained("/kaggle/working/genz-merged")
tokenizer.save_pretrained("/kaggle/working/genz-merged")
print("✅ Merged model saved")

✅ Adapter saved


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved


# 10. Download Model to Manually use Locally or upload on hugging Face


In [26]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
import shutil

shutil.make_archive("/kaggle/working/genz-merged-v2", "zip", "/kaggle/working/genz-merged-v2")

print("✅ genz-merged-v2.zip ready to download")

✅ genz-merged-v2.zip ready to download


In [27]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [28]:
from huggingface_hub import HfApi
api = HfApi()

# push adapter
api.create_repo("somendrew/genz-qwen-2.5-1.5B-adapter", exist_ok=True)
api.upload_folder(
    folder_path="/kaggle/working/genz-adapter",
    repo_id="somendrew/genz-qwen-2.5-1.5B-adapter",
    commit_message="Add GenZ LoRA adapter",
)
print("✅ Adapter pushed!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Adapter pushed!


In [29]:
# push merged model file by file (avoids timeout on 3GB)
api.create_repo("somendrew/genz-qwen-2.5-1.5B", exist_ok=True)
for filename in os.listdir("/kaggle/working/genz-merged-v2"):
    filepath = os.path.join("/kaggle/working/genz-merged-v2", filename)
    size = os.path.getsize(filepath) / 1024**2
    print(f"Uploading {filename} ({size:.1f} MB)...")
    api.upload_file(
        path_or_fileobj=filepath,
        path_in_repo=filename,
        repo_id="somendrew/genz-qwen-2.5-1.5B",
        commit_message=f"Upload {filename}",
    )
    print(f"✅ {filename} done!")

print("\n🎉 Both pushed to HuggingFace!")

Uploading config.json (0.0 MB)...
✅ config.json done!
Uploading chat_template.jinja (0.0 MB)...


No files have been modified since last commit. Skipping to prevent empty commit.


✅ chat_template.jinja done!
Uploading generation_config.json (0.0 MB)...
✅ generation_config.json done!
Uploading model.safetensors (2944.4 MB)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ tokenizer.json done!
Uploading tokenizer_config.json (0.0 MB)...


No files have been modified since last commit. Skipping to prevent empty commit.


✅ tokenizer_config.json done!

🎉 Both pushed to HuggingFace!
